<a href="https://colab.research.google.com/github/JiyadR/BigData26_A_2411533003_Jiyad-Rifqi-Pasaribu/blob/main/Praktikum5/BD_A_P05_2411533003_Jiyad_Rifqi_Pasaribu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

### **H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal**

Kode ini mengunduh dataset berita CSV dengan Pandas, lalu menjalankan web server lokal ([http://127.0.0.1:8000](http://127.0.0.1:8000)) secara background untuk menampilkan artikel dalam format HTML terpaginasi (100 berita/halaman) beserta penanganan rute khusus seperti /robots.txt.

In [ ]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom :", df_sumber.shape)
print("Kolom :", list(df_sumber.columns))

import http.server, urllib.parse, html

HALAMAN_UKURAN = 100        # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):          # matikan log
            pass
        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a '
                        f'href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>'
                    )
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Ukuran berkas : 16.1 MB
Baris, kolom : (91017, 4)
Kolom : ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


### **H-3. Crawling Portal Lokal: Latihan Aman**

Kode ini mengunduh file robots.txt dari situs (BASE) menggunakan requests dengan User-Agent khusus, lalu menganalisis ketentuannya menggunakan urllib.robotparser untuk mengecek dan mencetak status izin crawling (boleh atau tidak) pada jalur /berita?hal=1 dan /arsip-privat/.

In [ ]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


Kode ini mendefinisikan fungsi parse_halaman untuk mengekstrak data artikel (judul, URL, kategori, dan tanggal) dari HTML menggunakan BeautifulSoup, serta fungsi crawl_iter untuk melakukan crawling terstruktur yang mematuhi robots.txt, menerapkan mekanisme retry jika gagal, dan memberi jeda waktu antar-permintaan. Di akhir kode, proses crawling dijalankan untuk mengambil data dari halaman 1 hingga 3, lalu menampilkan total artikel yang diperoleh beserta 3 sampel data teratas.

In [ ]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


### **H-4. Crawling Berita Sungguhan dari Sitemap detik.com**
### **Langkah 1**

Kode ini mendefinisikan fungsi parse_sitemap_news untuk memproses berkas sitemap XML berita (khususnya format detik.com) menggunakan BeautifulSoup. Fungsi ini mengekstrak detail artikel seperti judul, URL, tanggal publikasi, dan kata kunci, serta memanfaatkan ekspresi reguler (regex) untuk mengambil kategori (subdomain) dan ID artikel langsung dari URL-nya sebelum mengembalikan data dalam bentuk daftar dictionary.

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else None,
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True) if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

### **Langkah 2**

Kode ini menyediakan sampel data XML sitemap berita (CONTOH_XML) untuk menguji fungsi parse_sitemap_news, lalu mencetak atribut artikel (judul, kategori, ID, tanggal) serta kata kuncinya, sekaligus membuktikan bahwa fungsi tersebut menghasilkan output yang identik baik menerima masukan bertipe string maupun bytes.

In [ ]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
<url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
        <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
        <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
        <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
        <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
</url>
<url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
        <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
        <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
        <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
</url>
<url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
        <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
        <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
        <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
</url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) ==
parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29T12:52:00+00:00'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29T12:50:00+00:00'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29T12:44:00+00:00'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


### **Langkah 3**

Kode ini mengatur proses crawling sitemap berita detik.com secara etis dengan mematuhi aturan robots.txt untuk setiap domain (diampu oleh boleh_diambil), menangani respons batas server serta jeda waktu 2 detik antar-permintaan (diampu oleh ambil_xml), dan mengumpulkan data artikel dari daftar sitemap pilihan hingga mencapai kuota yang ditentukan (maksimal 7 artikel per sitemap dan 20 artikel total) melalui fungsi crawl_detik.

In [ ]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: GANTI_DENGAN_EMAIL_ANDA)"
DELAY = 2.0                 # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7        # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20            # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content        # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

### **Langkah 4**

Kode ini menjalankan fungsi crawl_detik() dengan penanganan pengecualian (try-except), lalu menyimpan hasil crawling ke dalam berkas format JSONL (contoh_crawl/detik_sitemap.jsonl) dan mengonversinya menjadi DataFrame Pandas untuk menginspeksi kualitas data (mengecek kelengkapan judul/tanggal, duplikasi URL, sebaran kategori, serta pratinjau data teratas).

In [ ]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul    :", len(cek))
    print("Judul kosong       :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat       :", cek["url"].duplicated().sum())
    print("Tanggal kosong     :", cek["tanggal"].isna().sum())
    print("Kategori           :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul    : 20
Judul kosong       : 0
URL duplikat       : 6
Tanggal kosong     : 0
Kategori           : {'news': 7, 'sport': 6}
                                                                        judul kategori                   tanggal
                    Menyusuri Kota Tua Ala Turki, Ada Gereja Dijadikan Masjid     news 2026-11-09T03:46:00+07:00
Pria Mabuk Tenteng Golok Resahkan Warga Cibinong Bogor Berujung Dibawa Polisi     news 2026-11-09T01:53:00+07:00
       Kebakaran di Pool Damri Bandung Hanguskan 10 Bus, Menjalar ke 13 Rumah     news 2026-11-09T00:42:00+07:00


### **H-5. Channel ala Flume ke lapisan_mentah**

Kode ini menerapkan pola producer-consumer menggunakan multithreading dan antrean berbatas (queue.Queue) untuk memproses data crawling secara asinkron; fungsi source mengambil artikel dan memasukkannya ke antrean, sedangkan sink mengambil data dari antrean lalu menyimpannya ke dalam berkas-berkas JSONL berformat batch (hingga 5.000 entri per berkas) di folder lapisan_mentah serta mencetak statistik performa eksekusinya.

In [ ]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)             # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0
print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3],
"...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)

print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 56.1 detik untuk 911 halaman


### **H-6. Quality, Cleaning, dan Parquet Berpartisi (15 menit)**

Kode ini menggabungkan seluruh berkas JSONL dari folder mentah menjadi satu DataFrame Pandas, lalu menerapkan beberapa fungsi validasi kualitas data—seperti keabsahan panjang judul (3–30 kata), kategori yang dikenali, format tanggal, pola URL artikel, serta penanganan duplikasi—untuk menghitung dan menampilkan jumlah serta persentase baris data yang gagal memenuhi setiap kriteria uji tersebut.

In [ ]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"     : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata" : jml_kata.between(3, 30),
        "kategori dikenal" : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"     : pd.to_datetime(d["tanggal"], format="%Y-%m-%d",
errors="coerce").notna(),
        "url artikel valid" : d["url"].fillna("").str.match(POLA_ARTIKEL),
        "judul tidak duplikat"     : ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,}  ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0  ( 0.00%)
judul 3-30 kata          gagal:     78  ( 0.09%)
kategori dikenal         gagal:      0  ( 0.00%)
tanggal valid            gagal:      0  ( 0.00%)
url artikel valid        gagal:  5,640  ( 6.20%)
judul tidak duplikat     gagal:    235  ( 0.26%)


Kode ini melakukan pembersihan data (data cleaning) pada DataFrame mentah dengan merapikan spasi dan menghapus duplikasi judul, menyaring artikel dengan panjang judul 3–30 kata, mengonversi tanggal ke format datetime sekaligus mengekstrak kolom bulan (YYYY-MM), mengekstrak ID artikel dari URL, serta mengosongkan URL yang tidak valid sebelum mencetak rekapitulasi jumlah baris yang berhasil dibersihkan dan dibuang.

In [ ]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")  # kosong bila url rusak
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


Kode ini menguji dan membandingkan beberapa opsi strategi partisi data (berdasarkan kategori, bulan, atau gabungannya) untuk menyimpan DataFrame bersih ke format Parquet, kemudian menyimpan data kurasi akhirnya ke folder lapisan_terkurasi berpartisi kategori. Di bagian akhir, kode memuat dataset tersebut menggunakan pyarrow.dataset untuk mendemonstrasikan efisiensi partition pruning, yaitu membuktikan bahwa kueri filter (seperti kategori = finance) hanya membaca file partisi yang relevan tanpa perlu memindai seluruh file.

In [ ]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan dua pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


### **H-7. Query DuckDB dan Bentuk Dokumen**

Kode ini menggunakan DuckDB untuk membuat view berita dari berkas Parquet terpartisi Hive, lalu menjalankan kueri SQL untuk menghitung sebaran artikel per kategori dan tren bulanan kemunculan kata "corona". Di akhir, kode mengekstrak sampel artikel pertama dari DataFrame bersih beserta daftar token katanya menggunakan ekspresi reguler, lalu menampilkannya dalam format JSON berstruktur rapi.

In [ ]:
import duckdb

con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()

print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita
    WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan
    ORDER BY bulan
""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}

print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


### **H-8. Mini MapReduce: Kasus Kantin**

Kode ini mendemonstrasikan pola pemrograman MapReduce untuk menghitung total porsi pesanan dari tiga kasir, yang terdiri dari tahap Map (memecah string transaksi menjadi pasangan menu dan jumlah), Shuffle (mengelompokkan porsi berdasarkan jenis menu), dan Reduce (menjumlahkan total porsi per menu). Selain itu, kode ini juga mensimulasikan penggunaan Combiner untuk mengagregasi data secara lokal di setiap kasir terlebih dahulu guna mengoptimalkan dan mengurangi volume pasangan data yang dikirim ke tahap selanjutnya.

In [ ]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],  # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],  # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"],  # kasir C
]


def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))


def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))


# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())


setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


### **H-9. Word Count Terdistribusi: Partisi sebagai Input Split**

Kode ini menerapkan alur kerja MapReduce secara manual untuk menghitung frekuensi kata pada judul berita dari seluruh berkas partisi Parquet, dengan melakukan tokenisasi teks, pengelompokan (shuffle), penentuan kata teratas (sebelum dan sesudah penyaringan stopword), serta memverifikasi kebenaran hasilnya menggunakan collections.Counter konvensional.


In [ ]:
STOPWORD = set("""
    yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
    saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
    sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn
""".split())


def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())


# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))

for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f" {p.parent.name:<16} {n:>6,} judul")


def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]


hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)

hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")

print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f" {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common()
       if k not in STOPWORD]

print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f" {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
 kategori=finance 14,141 judul
 kategori=food     4,314 judul
 kategori=health   4,914 judul
 kategori=hot     16,290 judul
 kategori=inet     5,577 judul
 kategori=news    32,321 judul
 kategori=oto      4,375 judul
 kategori=sport    2,433 judul
 kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
 di          27,390
 corona      17,239
 ini         10,124
 yang         8,668
 dan          6,785
 ke           5,592
 tak          5,096
 dari         4,697
 jadi         4,486
 untuk        3,723

10 kata teratas (tanpa stopword):
 corona      17,239
 rp           3,154
 virus        3,075
 kasus        2,743
 polisi       2,732
 positif      2,682
 indonesia    2,504
 warga        2,495
 rumah        2,465
 jokowi       2,317

Sama dengan Counter biasa: True


### **H-10. Combiner dan Jumlah Data yang Berpindah**

Kode ini membandingkan performa eksekusi MapReduce secara sekuensial (1 proses) dan paralel (multiprocessing.Pool dengan 2 proses) baik tanpa maupun dengan combiner, untuk mengukur efisiensi waktu eksekusi serta pengurangan volume data yang dikirim, sekaligus memverifikasi bahwa hasil penggabungannya tetap identik.

In [ ]:
from multiprocessing import Pool


def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c


def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0


print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")

for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses)
        waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")

    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses)
        waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)

print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.509              825,601
1 proses, dengan combiner         0.549               74,009
2 proses, tanpa combiner          1.192              825,601
2 proses, dengan combiner         1.207               74,009

Hasil dengan combiner sama persis: True


### **H-11. Data Skew dan Beban Reducer**

Kode ini mensimulasikan fungsi partitioner berbasis hashing (zlib.crc32) untuk mendistribusikan data ke beberapa reducer (4, 16, dan 64), lalu menganalisis tingkat keseimbangan beban kerja (data skew) antar-reducer—baik dengan maupun tanpa combiner—serta melacak penumpukan beban yang disebabkan oleh kata-kata berfrekuensi tinggi (top keys).

In [ ]:
R = 4  # jumlah reducer


def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r


def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b


def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", " ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))


tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f" {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f" {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
   4 reducer: terberat 29.17% | bagian adil 25.00%
  16 reducer: terberat  8.82% | bagian adil  6.25%
  64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
 di        27,390 pasangan =  3.3% dari total, di reducer R0
 corona    17,239 pasangan =  2.1% dari total, di reducer R2
 ini       10,124 pasangan =  1.2% dari total, di reducer R0
 yang       8,668 pasangan =  1.0% dari total, di reducer R3
 dan        6,785 pasangan =  0.8% dari total, di reducer R3


### **H-12. Hadoop Streaming**

Kode ini membuat berkas masukan masukan.tsv dari data berita serta membuat dua skrip (mapper.py dan reducer.py) berbasis standard input/output gaya Hadoop Streaming untuk memproses tokenisasi dan agregasi frekuensi kata, lalu mencetak total baris masukan yang dihasilkan.

In [ ]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8"
)

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


Kode ini menguji skrip mapper.py pada 3 baris pertama masukan, lalu mengukur waktu eksekusi seluruh pipeline MapReduce berbasis Unix pipe (mengalirkan data melalui mapper, sort, dan reducer ke hasil_pipe.txt), menghitung total kata unik, serta menampilkan 5 kata dengan frekuensi kemunculan tertinggi.

In [ ]:
!head -n 3 masukan.tsv | python3 mapper.py
!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m3.067s
user	0m2.386s
sys	0m0.119s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


Kode ini membaca berkas hasil_pipe.txt hasil eksekusi pipeline Unix ke dalam dictionary Python, lalu memverifikasi kebenaran datanya dengan mencocokkan apakah hasilnya identik dengan variabel hasil_semua yang diperoleh dari proses MapReduce sebelumnya.

In [ ]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)

print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


Kode di atas mengunduh serta memasang OpenJDK 11 dan Apache Hadoop 3.3.6 untuk mengeksekusi job Hadoop Streaming secara lokal (standalone mode) menggunakan berkas mapper.py dan reducer.py pada masukan masukan.tsv. Setelah proses eksekusi selesai dan hasilnya disimpan ke dalam direktori keluaran_hadoop, kode menjalankan perintah diff untuk membandingkan luaran Hadoop (keluaran_hadoop/part-00000) dengan hasil simulasi Unix pipe (hasil_pipe.txt) guna memverifikasi bahwa keduanya menghasilkan keluaran yang persis sama.

In [ ]:
%%bash
sudo apt-get install -y openjdk-11-jdk-headless

wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
tar -xzf hadoop-3.3.6.tar.gz

export HADOOP_HOME=$PWD/hadoop-3.3.6
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64

rm -rf keluaran_hadoop

$HADOOP_HOME/bin/hadoop jar \
    $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
    -files mapper.py,reducer.py \
    -mapper "python3 mapper.py" \
    -reducer "python3 reducer.py" \
    -input masukan.tsv \
    -output keluaran_hadoop

ls keluaran_hadoop

diff <(LC_ALL=C sort hasil_pipe.txt) keluaran_hadoop/part-00000 && echo "SAMA"

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  openjdk-11-jre-headless
Suggested packages:
  openjdk-11-demo openjdk-11-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-11-jdk-headless openjdk-11-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 0 not upgraded.
Need to get 116 MB of archives.
After this operation, 258 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 openjdk-11-jre-headless amd64 11.0.32.1+1-1ubuntu1~24.04 [42.3 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 openjdk-11-jdk-headless amd64 11.0.32.1+1-1ubuntu1~24.04 [73.7 MB]
Fetched 116 MB in 4s (30.1 MB/s)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 1

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
2026-11-08 22:03:33,467 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-11-08 22:03:33,819 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-11-08 22:03:33,819 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-11-08 22:03:33,861 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-11-08 22:03:34,304 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-08 22:03:34,343 INFO mapreduce.JobSubmitter: number of splits:1

### **H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman**

Kode ini menghitung frekuensi kemunculan kata non-stopword per kategori berita menggunakan kunci gabungan (kategori, kata), lalu mencetak 5 kata teratas untuk tiap kategori. Setelah itu, hasil analisis disimpan ke berkas Parquet di folder siap_pakai dan diekspor ke basis data relasional SQLite (laporan.db) sebagai simulasi proses Sqoop export, kemudian diverifikasi lewat kueri SQL untuk mempastikan jumlah baris dan menampilkan 3 kata teratas pada kategori health.

In [ ]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c


gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))

for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3

SIAP = Path("siap_pakai")
SIAP.mkdir(exist_ok=True)

baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""
    SELECT kata, jumlah FROM kata_per_kategori
    WHERE kategori = 'health'
    ORDER BY jumlah DESC
    LIMIT 3
""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


### **Studi Kasus**

In [ ]:

# ============================================================
# M. STUDI KASUS
# Ringkasan Pemberitaan Portal Berita Paruh Pertama Tahun 2020
# Menggunakan hasil H-7 dan H-13
# ============================================================

from collections import defaultdict

# ------------------------------------------------------------
# 1. MENGAMBIL TREN BULANAN DARI H-7
# ------------------------------------------------------------

tren_corona = con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita
    WHERE lower(judul) LIKE '%corona%'
      AND bulan BETWEEN '2020-01' AND '2020-06'
    GROUP BY bulan
    ORDER BY bulan
""").df()

tren_corona["bulan"] = tren_corona["bulan"].astype(str)

print("TREN PEMBERITAAN CORONA (H-7)")
print(tren_corona.to_string(index=False))


# ------------------------------------------------------------
# 2. MENGHITUNG KOSAKATA UNIK PER KATEGORI DARI H-13
# ------------------------------------------------------------

kosakata_unik = defaultdict(set)

for (kategori, kata), jumlah in gabung.items():
    if kata not in STOPWORD:
        kosakata_unik[kategori].add(kata)

daftar_kategori = sorted(bersih["kategori"].dropna().unique())

jumlah_judul = bersih.groupby("kategori").size().to_dict()

rekap = []

for kategori in daftar_kategori:
    rekap.append({
        "kategori": kategori,
        "jumlah_judul": int(jumlah_judul[kategori]),
        "frekuensi_corona": int(gabung.get((kategori, "corona"), 0)),
        "kata_unik": len(kosakata_unik[kategori])
    })

rekap_kategori = pd.DataFrame(rekap)

print("\nRINGKASAN PEMBERITAAN PER KATEGORI (H-13)")
print(
    rekap_kategori
    .sort_values("frekuensi_corona", ascending=False)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 3. MENENTUKAN KATEGORI TERBANYAK DAN PALING BERAGAM
# ------------------------------------------------------------

# Kategori dengan frekuensi kata corona tertinggi
dominan = rekap_kategori.loc[
    rekap_kategori["frekuensi_corona"].idxmax()
]

kategori_corona = dominan["kategori"]
jumlah_corona = int(dominan["frekuensi_corona"])

# Kategori dengan jumlah kosakata unik terbanyak
beragam = rekap_kategori.loc[
    rekap_kategori["kata_unik"].idxmax()
]

kategori_beragam = beragam["kategori"]
jumlah_unik = int(beragam["kata_unik"])


# Mengambil 3 kata teratas untuk kategori tertentu
def tiga_kata_teratas(kategori):
    pasangan = [
        (jumlah, kata)
        for (kat, kata), jumlah in gabung.items()
        if kat == kategori
    ]

    return [
        kata for jumlah, kata
        in sorted(pasangan, reverse=True)[:3]
    ]


top3 = tiga_kata_teratas(kategori_corona)


# ------------------------------------------------------------
# 4. MENYIAPKAN ANGKA TREN BULANAN
# ------------------------------------------------------------

angka_bulan = {
    row["bulan"]: int(row["judul_corona"])
    for _, row in tren_corona.iterrows()
}

nama_bulan = {
    "2020-01": "Januari",
    "2020-02": "Februari",
    "2020-03": "Maret",
    "2020-04": "April",
    "2020-05": "Mei",
    "2020-06": "Juni"
}

def format_angka(nilai):
    return f"{int(nilai):,}".replace(",", ".")


bulan_puncak = max(angka_bulan, key=angka_bulan.get)

jan = format_angka(angka_bulan["2020-01"])
feb = format_angka(angka_bulan["2020-02"])
apr = format_angka(angka_bulan["2020-04"])
mei = format_angka(angka_bulan["2020-05"])
jun = format_angka(angka_bulan["2020-06"])

puncak = format_angka(angka_bulan[bulan_puncak])


# ------------------------------------------------------------
# 5. MEMBUAT RINGKASAN STUDI KASUS (150-200 KATA)
# ------------------------------------------------------------

ringkasan = f"""
Pada paruh pertama 2020, perhatian portal berita terhadap
corona meningkat dari {jan} judul pada Januari menjadi
{feb} pada Februari, lalu mencapai puncak pada
{nama_bulan[bulan_puncak]} dengan {puncak} judul.
Setelah itu jumlahnya turun menjadi {apr} pada April
dan {mei} pada Mei. Juni mencatat {jun} judul, tetapi
angka ini hanya mencakup 13 hari sehingga belum mewakili
satu bulan penuh.

Berdasarkan frekuensi kata H-13, kategori {kategori_corona}
paling sering menggunakan kata "corona", sebanyak
{format_angka(jumlah_corona)} kali. Tiga kata teratas
kategori tersebut adalah "{top3[0]}", "{top3[1]}",
dan "{top3[2]}". Untuk keragaman kosakata judul,
kategori {kategori_beragam} memiliki istilah non-stopword
unik terbanyak, yaitu {format_angka(jumlah_unik)}
kata berbeda.

Istilah teratas pada kategori lain memperlihatkan sudut
pemberitaan yang berbeda: health menonjolkan istilah
"virus", "pasien", dan "meninggal"; finance memuat
"rp", "ri", dan "harga"; sedangkan travel menampilkan
"wisata" dan "turis". Temuan ini menunjukkan bahwa isu
corona tidak hanya hadir sebagai isu kesehatan, tetapi
juga bersinggungan dengan berita umum, ekonomi, dan
aktivitas masyarakat.

Namun, analisis hanya menggunakan judul, bukan isi artikel
lengkap. Frekuensi kata tidak menunjukkan kedalaman atau
konteks pembahasan. Jumlah kosakata unik juga dapat
dipengaruhi banyaknya judul dalam kategori. Selain itu,
data Juni belum lengkap.
"""

# ------------------------------------------------------------
# 6. MENAMPILKAN HASIL AKHIR DAN JUMLAH KATA
# ------------------------------------------------------------

jumlah_kata = len(ringkasan.split())

print("\n" + "=" * 65)
print("M. STUDI KASUS - RINGKASAN PEMBERITAAN PORTAL BERITA")
print("=" * 65)

print(ringkasan.strip())

print("\n" + "=" * 65)
print("Jumlah kata:", jumlah_kata)
print("Kategori corona terbanyak:", kategori_corona)
print("Kategori kosakata paling beragam:", kategori_beragam)
print("Puncak pemberitaan:", nama_bulan[bulan_puncak], 2020)

if 150 <= jumlah_kata <= 200:
    print("STATUS: Ringkasan memenuhi ketentuan 150-200 kata.")
else:
    print("PERHATIAN: Sesuaikan ringkasan agar menjadi 150-200 kata.")

print("=" * 65)


TREN PEMBERITAAN CORONA (H-7)
  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

RINGKASAN PEMBERITAAN PER KATEGORI (H-13)
kategori  jumlah_judul  frekuensi_corona  kata_unik
    news         32321              7539      16412
  health          4914              2590       5165
 finance         14141              2470       8165
     hot         16290              1658      13627
  travel          6453              1023       7614
    inet          5577               845       6701
     oto          4375               576       5386
    food          4314               310       6500
   sport          2433               228       4031

M. STUDI KASUS - RINGKASAN PEMBERITAAN PORTAL BERITA
Pada paruh pertama 2020, perhatian portal berita terhadap
corona meningkat dari 385 judul pada Januari menjadi
1.077 pada Februari, lalu mencapai puncak pada
Maret dengan 6.994 judul.
Setelah itu jum

### **Latihan**

Kode ini memperluas daftar stopword dengan menambahkan kata-kata spesifik domain berita (seperti "rp", "ri", "foto", "indonesia", dan "the"), lalu membandingkan frekuensi 5 kata teratas per kategori sebelum dan sesudah penyaringan tambahan tersebut. Melalui fungsi hitung_stopword_per_kategori(), data judul dibaca dari partisi Parquet untuk di-tokenisasi dan dihitung ulang jumlah kemunculan katanya, sehingga terlihat bagaimana eliminasi kata-kata umum domain meningkatkan kualitas serta keunikan topik utama pada setiap kategori.

In [ ]:

# ============================================================
# N.1 STOPWORD DOMAIN
# ============================================================

STOPWORD_LAMA = set(STOPWORD)

STOPWORD_TAMBAHAN = {
    "rp",
    "ri",
    "the",
    "foto",
    "indonesia"
}

STOPWORD_UJI = STOPWORD_LAMA | STOPWORD_TAMBAHAN

print("Lima stopword tambahan:")
print(sorted(STOPWORD_TAMBAHAN))


# Menghitung frekuensi kata setiap kategori
def hitung_stopword_per_kategori(daftar_stopword):

    hasil = {}

    for path in splits:

        df_tmp = pd.read_parquet(
            path,
            columns=["judul"]
        )

        kategori = path.parent.name.split("=")[1]

        if kategori not in hasil:
            hasil[kategori] = collections.Counter()

        for judul in df_tmp["judul"].dropna():

            kata = [
                k for k in tokenisasi(judul)
                if k not in daftar_stopword
            ]

            hasil[kategori].update(kata)

    return hasil


# Perhitungan sebelum dan sesudah
hasil_stopword_lama = hitung_stopword_per_kategori(
    STOPWORD_LAMA
)

hasil_stopword_baru = hitung_stopword_per_kategori(
    STOPWORD_UJI
)


# Menampilkan perbandingan Top 5
print("\nPERBANDINGAN 5 KATA TERATAS PER KATEGORI")

for kategori in sorted(hasil_stopword_lama):

    top_lama = hasil_stopword_lama[kategori].most_common(5)
    top_baru = hasil_stopword_baru[kategori].most_common(5)

    print("\nKategori:", kategori)

    print("Sebelum:")
    print(", ".join(
        f"{kata} ({jumlah:,})"
        for kata, jumlah in top_lama
    ))

    print("Sesudah:")
    print(", ".join(
        f"{kata} ({jumlah:,})"
        for kata, jumlah in top_baru
    ))


print("\nSTOPWORD asli tetap:", len(STOPWORD), "kata")
print("STOPWORD eksperimen:", len(STOPWORD_UJI), "kata tambahan 5")


Lima stopword tambahan:
['foto', 'indonesia', 'ri', 'rp', 'the']

PERBANDINGAN 5 KATA TERATAS PER KATEGORI

Kategori: finance
Sebelum:
corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
Sesudah:
corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)

Kategori: food
Sebelum:
makanan (699), makan (539), foto (458), galeri (438), enak (358)
Sesudah:
makanan (699), makan (539), galeri (438), enak (358), bikin (343)

Kategori: health
Sebelum:
corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
Sesudah:
corona (2,590), virus (1,040), pasien (471), meninggal (413), kasus (364)

Kategori: hot
Sebelum:
corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
Sesudah:
corona (1,658), anak (793), rumah (561), meninggal (545), film (489)

Kategori: inet
Sebelum:
corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
Sesudah:
corona (845), virus (341), ponsel (309), bikin (253), google (221)

Kategori: news
Sebelum:
cor

Kode ini menganalisis tren kata kunci judul berita per bulan dengan memetakan pasangan kunci (bulan, kata) dan menghitung frekuensi kata non-stopword. Selanjutnya, kode menampilkan 10 kata teratas untuk setiap bulan serta membandingkan pergerakan topik antara bulan Januari dan Mei 2020—termasuk mengidentifikasi kata kunci yang baru masuk, keluar, dan konsisten bertahan di jajaran 10 besar beserta perubahan frekuensi kemunculannya.

In [ ]:

# ============================================================
# N.2 ANALISIS KATA PER BULAN
# ============================================================

kata_per_bulan_uji = collections.defaultdict(
    collections.Counter
)

# Mapper dengan key (bulan, kata)
def mapper_per_bulan(path):

    df_tmp = pd.read_parquet(
        path,
        columns=["judul", "bulan"]
    )

    hasil = collections.Counter()

    for _, row in df_tmp.iterrows():

        bulan = row["bulan"]

        if pd.isna(bulan):
            continue

        bulan = str(bulan)

        for kata in tokenisasi(row["judul"]):

            if kata not in STOPWORD:
                hasil[(bulan, kata)] += 1

    return hasil


# Map dan Reduce
for path in splits:

    hasil_mapper_bulan = mapper_per_bulan(path)

    for (bulan, kata), jumlah in hasil_mapper_bulan.items():
        kata_per_bulan_uji[bulan][kata] += jumlah


# Menampilkan Top 10 setiap bulan
print("10 KATA TERATAS SETIAP BULAN")

for bulan in sorted(kata_per_bulan_uji):

    print("\nBulan:", bulan)

    for kata, jumlah in kata_per_bulan_uji[bulan].most_common(10):
        print(f"{kata:<15} {jumlah:>8,}")


# Membandingkan Januari dan Mei
top_januari = kata_per_bulan_uji[
    "2020-01"
].most_common(10)

top_mei = kata_per_bulan_uji[
    "2020-05"
].most_common(10)

kata_januari = {kata for kata, jumlah in top_januari}
kata_mei = {kata for kata, jumlah in top_mei}

kata_masuk = kata_mei - kata_januari
kata_keluar = kata_januari - kata_mei
kata_tetap = kata_januari & kata_mei

print("\nPERUBAHAN JANUARI KE MEI")

print("\nKata yang masuk Top 10 pada Mei:")
print(sorted(kata_masuk))

print("\nKata yang keluar dari Top 10 pada Mei:")
print(sorted(kata_keluar))

print("\nKata yang tetap berada di Top 10:")
print(sorted(kata_tetap))


# Perubahan jumlah kemunculan kata yang tetap berada di Top 10
print("\nPERUBAHAN FREKUENSI KATA YANG TETAP")

for kata in sorted(kata_tetap):

    awal = kata_per_bulan_uji["2020-01"][kata]
    akhir = kata_per_bulan_uji["2020-05"][kata]

    print(
        f"{kata:<15} Januari: {awal:>7,} | "
        f"Mei: {akhir:>7,} | "
        f"Perubahan: {akhir - awal:+,}"
    )


10 KATA TERATAS SETIAP BULAN

Bulan: 2020-01
banjir             1,220
rp                   616
polisi               554
jokowi               516
tahun                493
baru                 475
mobil                428
minta                427
kpk                  415
indonesia            410

Bulan: 2020-02
corona             1,076
polisi               630
virus                596
banjir               572
rp                   572
anak                 453
ri                   433
indonesia            415
jakarta              407
wni                  404

Bulan: 2020-03
corona             6,986
virus              1,340
positif              925
pasien               749
rumah                695
cegah                665
rp                   606
indonesia            585
jokowi               537
kasus                524

Bulan: 2020-04
corona             4,914
positif              828
pandemi              706
psbb                 689
warga                609
meninggal            577
rumah  

Kode ini menerapkan analisis bigram (pasangan dua kata berurutan) menggunakan pola MapReduce untuk membandingkan efisiensi pemrosesan tanpa dan dengan combiner. Melalui mapper, setiap judul berita dipecah menjadi pasangan kata berurutan, lalu diukur sejauh mana combiner berbasis collections.Counter dapat mengurangi volume data yang dikirim ke tahap shuffle. Selain menampilkan 15 bigram dengan frekuensi kemunculan tertinggi, kode ini memverifikasi bahwa pengolahan lokal pada combiner berhasil menghemat pengiriman data tanpa mengubah keakuratan hasil akhirnya.

In [ ]:

# ============================================================
# N.3 EKSPERIMEN BIGRAM
# ============================================================

# Mapper tanpa combiner
def mapper_bigram_tanpa_combiner(path):

    df_tmp = pd.read_parquet(
        path,
        columns=["judul"]
    )

    hasil = []

    for judul in df_tmp["judul"].dropna():

        kata = tokenisasi(judul)

        # Membuat pasangan dua kata berurutan
        pasangan = zip(kata, kata[1:])

        for a, b in pasangan:
            hasil.append(((a, b), 1))

    return hasil


# Mapper dengan combiner
def mapper_bigram_dengan_combiner(path):

    df_tmp = pd.read_parquet(
        path,
        columns=["judul"]
    )

    hasil = collections.Counter()

    for judul in df_tmp["judul"].dropna():

        kata = tokenisasi(judul)

        pasangan = zip(kata, kata[1:])

        hasil.update(pasangan)

    return hasil


# Menjalankan seluruh mapper tanpa combiner
bigram_tanpa_combiner = [
    mapper_bigram_tanpa_combiner(p)
    for p in splits
]


# Menjalankan seluruh mapper dengan combiner
bigram_dengan_combiner = [
    mapper_bigram_dengan_combiner(p)
    for p in splits
]


# Menghitung jumlah pasangan yang dikirim
jumlah_bigram_awal = sum(
    len(x) for x in bigram_tanpa_combiner
)

jumlah_bigram_combiner = sum(
    len(x) for x in bigram_dengan_combiner
)


# Proses Reduce tanpa combiner
hasil_reduce_bigram_lama = collections.Counter()

for mapper in bigram_tanpa_combiner:

    hasil_reduce_bigram_lama.update(
        pasangan for pasangan, jumlah in mapper
    )


# Proses Reduce dengan combiner
hasil_reduce_bigram_baru = collections.Counter()

for mapper in bigram_dengan_combiner:
    hasil_reduce_bigram_baru.update(mapper)


# Menampilkan hasil
print("PERBANDINGAN JUMLAH PASANGAN BIGRAM")

print(
    "Tanpa combiner :",
    f"{jumlah_bigram_awal:,}"
)

print(
    "Dengan combiner:",
    f"{jumlah_bigram_combiner:,}"
)

print(
    "Bigram berhasil dikurangi:",
    f"{jumlah_bigram_awal - jumlah_bigram_combiner:,}"
)


# Menampilkan bigram teratas
print("\n15 BIGRAM PALING SERING MUNCUL")

for (kata1, kata2), jumlah in (
    hasil_reduce_bigram_baru.most_common(15)
):

    print(
        f"{kata1} {kata2:<25} {jumlah:>8,}"
    )


# Verifikasi
print(
    "\nHasil tanpa dan dengan combiner sama:",
    hasil_reduce_bigram_lama == hasil_reduce_bigram_baru
)


PERBANDINGAN JUMLAH PASANGAN BIGRAM
Tanpa combiner : 734,783
Dengan combiner: 411,217
Bigram berhasil dikurangi: 323,566

15 BIGRAM PALING SERING MUNCUL
virus corona                       2,915
corona di                           2,178
positif corona                       1,804
di tengah                       1,258
pandemi corona                       1,075
di rumah                        1,047
new normal                         917
di indonesia                      871
rp juta                           700
hari ini                            627
rp m                              606
pasien corona                         600
di jakarta                        584
cegah corona                         583
rp t                              549

Hasil tanpa dan dengan combiner sama: True


Kode ini mensimulasikan dan menganalisis tingkat ketimpangan beban kerja (data skew) pada variasi jumlah reducer (2, 8, dan 32) menggunakan fungsi partitioner berbasis hashing (zlib.crc32). Dengan membandingkan skenario tanpa dan dengan combiner, kode ini mengukur porsi pemrosesan pada reducer terberat lalu menghitung rasionya terhadap alokasi ideal (bagian adil), yang kemudian ditampilkan secara terstruktur menggunakan tabel DataFrame Pandas.

In [ ]:

# ============================================================
# N.4 EKSPERIMEN JUMLAH REDUCER
# ============================================================

# Menggunakan hasil H-9 dan fungsi mapper H-10
# Variabel asli tidak ditimpa

mapper_uji_tanpa = [
    mapper_split(p)
    for p in splits
]

mapper_uji_combiner = [
    mapper_combiner(p)
    for p in splits
]


# Fungsi partitioner
def partitioner_uji(kata, jumlah_reducer):

    return (
        zlib.crc32(kata.encode("utf-8"))
        % jumlah_reducer
    )


# Fungsi menghitung beban reducer
def hitung_beban_uji(data_mapper, jumlah_reducer):

    beban = [0] * jumlah_reducer

    for hasil_mapper in data_mapper:

        if isinstance(hasil_mapper, collections.Counter):
            pasangan = hasil_mapper.items()
        else:
            pasangan = hasil_mapper

        for kata, jumlah in pasangan:

            nomor = partitioner_uji(
                kata,
                jumlah_reducer
            )

            beban[nomor] += 1

    return beban


# Menjalankan pengujian
hasil_reducer_uji = []

for R_UJI in (2, 8, 32):

    skenario = [
        ("Tanpa Combiner", mapper_uji_tanpa),
        ("Dengan Combiner", mapper_uji_combiner)
    ]

    for nama, data_mapper in skenario:

        beban = hitung_beban_uji(
            data_mapper,
            R_UJI
        )

        total = sum(beban)
        terberat = max(beban)

        bagian_terberat = (
            terberat / total * 100
            if total else 0
        )

        bagian_adil = 100 / R_UJI

        rasio = (
            bagian_terberat / bagian_adil
            if bagian_adil else 0
        )

        hasil_reducer_uji.append({
            "Reducer": R_UJI,
            "Skenario": nama,
            "Beban Terberat": terberat,
            "Terberat (%)": round(bagian_terberat, 2),
            "Bagian Adil (%)": round(bagian_adil, 2),
            "Rasio terhadap Adil": round(rasio, 2)
        })


# Menampilkan tabel perbandingan
tabel_reducer_uji = pd.DataFrame(
    hasil_reducer_uji
)

print("PERBANDINGAN BEBAN REDUCER")

print(
    tabel_reducer_uji.to_string(index=False)
)


print("\nKeterangan:")
print("Rasio 1,00 = beban terberat sama dengan bagian ideal.")
print("Rasio > 1  = beban terberat lebih besar daripada bagian ideal.")


PERBANDINGAN BEBAN REDUCER
 Reducer        Skenario  Beban Terberat  Terberat (%)  Bagian Adil (%)  Rasio terhadap Adil
       2  Tanpa Combiner          431563         52.27            50.00                 1.05
       2 Dengan Combiner           37178         50.23            50.00                 1.00
       8  Tanpa Combiner          138978         16.83            12.50                 1.35
       8 Dengan Combiner            9512         12.85            12.50                 1.03
      32  Tanpa Combiner           48073          5.82             3.12                 1.86
      32 Dengan Combiner            2545          3.44             3.12                 1.10

Keterangan:
Rasio 1,00 = beban terberat sama dengan bagian ideal.
Rasio > 1  = beban terberat lebih besar daripada bagian ideal.


Kode ini mensimulasikan proses crawling ulang pada 50 halaman berita dengan jeda 0,05 detik untuk mengukur waktu eksekusi aktual serta menghitung rata-rata overhead pemrosesan per halaman. Dari hasil sampel tersebut, kode mengekstrapolasi data untuk mengestimasi total waktu yang dibutuhkan jika pemrosesan dilakukan pada skala penuh 911 halaman dengan jeda standar 1,0 detik, lalu menampilkannya dalam format menit dan detik yang rapi.

In [ ]:

# ============================================================
# N.5 EKSPERIMEN CRAWL ULANG
# ============================================================

HALAMAN_UJI = min(50, N_HALAMAN)
DELAY_UJI = 0.05

daftar_halaman_uji = range(
    1,
    HALAMAN_UJI + 1
)

# Mulai pengukuran
waktu_mulai = time.perf_counter()

hasil_crawl_50 = list(
    crawl_iter(
        BASE,
        daftar_halaman_uji,
        delay=DELAY_UJI
    )
)

waktu_50 = time.perf_counter() - waktu_mulai


# Menghitung estimasi 911 halaman
JUMLAH_HALAMAN_911 = 911
DELAY_911 = 1.0

# Perkiraan waktu pemrosesan selain jeda
overhead_per_halaman = max(
    0,
    (
        waktu_50 - (HALAMAN_UJI * DELAY_UJI)
    ) / HALAMAN_UJI
)

# Waktu jeda saja
waktu_jeda_911 = (
    JUMLAH_HALAMAN_911 * DELAY_911
)

# Estimasi keseluruhan
estimasi_waktu_911 = JUMLAH_HALAMAN_911 * (
    DELAY_911 + overhead_per_halaman
)


# Fungsi format waktu
def format_waktu_uji(detik):

    menit = int(detik // 60)
    sisa = detik % 60

    return f"{menit} menit {sisa:.2f} detik"


# Menampilkan hasil
print("HASIL CRAWL 50 HALAMAN")

print("Jumlah halaman:", HALAMAN_UJI)
print("Jumlah artikel:", len(hasil_crawl_50))
print("Jeda:", DELAY_UJI, "detik")

print(
    "Waktu aktual:",
    format_waktu_uji(waktu_50)
)


print("\nESTIMASI CRAWL 911 HALAMAN")

print(
    "Waktu jeda saja:",
    format_waktu_uji(waktu_jeda_911)
)

print(
    "Estimasi termasuk waktu pemrosesan:",
    format_waktu_uji(estimasi_waktu_911)
)

print(
    "\nEstimasi overhead per halaman:",
    round(overhead_per_halaman, 4),
    "detik"
)


HASIL CRAWL 50 HALAMAN
Jumlah halaman: 50
Jumlah artikel: 5000
Jeda: 0.05 detik
Waktu aktual: 0 menit 4.84 detik

ESTIMASI CRAWL 911 HALAMAN
Waktu jeda saja: 15 menit 11.00 detik
Estimasi termasuk waktu pemrosesan: 15 menit 53.71 detik

Estimasi overhead per halaman: 0.0469 detik


Kode ini mengeksplorasi ekstraksi berita dari kategori baru dengan membaca sitemap index (sitemap.xml) untuk menemukan sitemap news alternatif yang belum diproses, lalu mengganti salah satu sitemap lama dalam daftar eksperimen. Selanjutnya, skrip melakukan crawling sampel artikel (maksimal 20 judul dengan pembatasan 7 judul per sitemap dan eliminasi duplikasi URL), menormalisasi format tanggal ke standar YYYY-MM-DD, serta mengevaluasi kelayakan data menggunakan fungsi aturan_kualitas untuk menampilkan sebaran kategori berita, contoh artikel, dan tabel ringkasan tingkat kelulusan kualitas data.

In [ ]:

# ============================================================
# N.6 EKSPERIMEN SITEMAP BAGIAN LAIN
# ============================================================

import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

# Persiapan session
sesi_sitemap_uji = requests.Session()
sesi_sitemap_uji.headers["User-Agent"] = UA_DETIK

# Sitemap index yang akan diperiksa
DAFTAR_INDEX_UJI = [
    "https://www.detik.com/sitemap.xml",
    "https://detik.com/sitemap.xml"
]

xml_index_uji = None
url_index_berhasil = None

print("MENCARI SITEMAP INDEX")

for alamat_index in DAFTAR_INDEX_UJI:

    try:

        print("\nMemeriksa:", alamat_index)

        xml_index_uji = ambil_xml(
            alamat_index,
            sesi_sitemap_uji
        )

        url_index_berhasil = alamat_index
        break

    except Exception as e:

        print(
            "Gagal:",
            type(e).__name__,
            "-",
            e
        )


# Menyiapkan hasil crawl
hasil_crawl_sitemap_lain = []


if xml_index_uji is None:

    print("\nSitemap index tidak berhasil diakses.")

else:

    print("\nSitemap index berhasil:", url_index_berhasil)

    # Membaca daftar alamat sitemap
    soup_index = BeautifulSoup(
        xml_index_uji,
        "xml"
    )

    semua_url_sitemap = [
        loc.get_text(strip=True)
        for loc in soup_index.find_all("loc")
    ]

    # Mengambil kategori dari sitemap yang sudah digunakan
    kategori_lama = {
        urlparse(url).hostname.split(".")[0]
        for url in SITEMAP_DIPILIH
    }

    # Mencari sitemap_news dari kategori lain
    kandidat_sitemap = []

    for url in semua_url_sitemap:

        host = urlparse(url).hostname

        if not host:
            continue

        bagian = host.split(".")[0].lower()

        if (
            url.rstrip("/").endswith("sitemap_news.xml")
            and bagian in KATEGORI_DIKENAL
            and bagian not in kategori_lama
        ):

            if url not in kandidat_sitemap:
                kandidat_sitemap.append(url)


    print("\nSITEMAP NEWS BAGIAN LAIN YANG DITEMUKAN")

    for url in kandidat_sitemap:
        print(url)


    if not kandidat_sitemap:

        print(
            "\nTidak ditemukan sitemap_news kategori lain "
            "dalam sitemap index yang berhasil dibaca."
        )

    else:

        # Menggunakan kandidat pertama
        sitemap_alternatif = kandidat_sitemap[0]

        # Menyalin daftar sitemap asli
        SITEMAP_DIPILIH_UJI = list(SITEMAP_DIPILIH)

        # Mengganti satu sitemap, yaitu alamat terakhir
        sitemap_sebelumnya = SITEMAP_DIPILIH_UJI[-1]

        SITEMAP_DIPILIH_UJI[-1] = sitemap_alternatif

        print("\nPERUBAHAN SITEMAP")
        print("Sebelum:", sitemap_sebelumnya)
        print("Sesudah:", sitemap_alternatif)

        print("\nDaftar sitemap eksperimen:")

        for url in SITEMAP_DIPILIH_UJI:
            print(url)


        # ----------------------------------------------------
        # CRAWLING MAKSIMAL 20 JUDUL
        # Maksimal 7 judul dari setiap sitemap
        # ----------------------------------------------------

        url_sudah_ada = set()

        for url in SITEMAP_DIPILIH_UJI:

            if len(hasil_crawl_sitemap_lain) >= BATAS_TOTAL:
                break

            try:

                print("\nMengambil:", url)

                xml_data = ambil_xml(
                    url,
                    sesi_sitemap_uji
                )

                artikel = parse_sitemap_news(xml_data)

                batas = min(
                    BATAS_PER_BAGIAN,
                    BATAS_TOTAL - len(hasil_crawl_sitemap_lain)
                )

                jumlah_sebelum = len(hasil_crawl_sitemap_lain)

                for data in artikel:

                    if len(hasil_crawl_sitemap_lain) >= (
                        jumlah_sebelum + batas
                    ):
                        break

                    if data["url"] in url_sudah_ada:
                        continue

                    hasil_crawl_sitemap_lain.append(data)
                    url_sudah_ada.add(data["url"])


            except Exception as e:

                print(
                    "Sitemap gagal diambil:",
                    type(e).__name__,
                    "-",
                    e
                )


        print("\nJUMLAH JUDUL HASIL CRAWLING")

        print(
            len(hasil_crawl_sitemap_lain),
            "dari maksimal",
            BATAS_TOTAL,
            "judul"
        )


        # ----------------------------------------------------
        # MENAMPILKAN KATEGORI
        # ----------------------------------------------------

        if hasil_crawl_sitemap_lain:

            df_sitemap_alt = pd.DataFrame(
                hasil_crawl_sitemap_lain
            )

            print("\nSEBARAN KATEGORI")

            print(
                df_sitemap_alt["kategori"]
                .value_counts(dropna=False)
                .to_string()
            )

            print("\nCONTOH JUDUL HASIL CRAWLING")

            print(
                df_sitemap_alt[
                    ["judul", "kategori", "tanggal"]
                ].to_string(index=False)
            )


            # ------------------------------------------------
            # MENERAPKAN ATURAN KUALITAS H-6
            # ------------------------------------------------

            df_kualitas_uji = df_sitemap_alt.copy()

            # Sitemap menggunakan ISO 8601, sedangkan aturan
            # kualitas H-6 menggunakan YYYY-MM-DD.
            df_kualitas_uji["tanggal"] = (
                pd.to_datetime(
                    df_kualitas_uji["tanggal"],
                    errors="coerce",
                    utc=True
                )
                .dt.strftime("%Y-%m-%d")
            )


            hasil_aturan_uji = aturan_kualitas(
                df_kualitas_uji
            )


            # Menghitung jumlah data gagal setiap aturan
            laporan_kualitas_uji = []

            for nama, lulus in hasil_aturan_uji.items():

                gagal = int((~lulus).sum())
                total = len(lulus)

                persentase = (
                    gagal / total * 100
                    if total else 0
                )

                laporan_kualitas_uji.append({
                    "Aturan Kualitas": nama,
                    "Gagal": gagal,
                    "Lulus": total - gagal,
                    "Total": total,
                    "Gagal (%)": round(persentase, 2)
                })


            tabel_kualitas_uji = pd.DataFrame(
                laporan_kualitas_uji
            )

            print("\nHASIL PENGUJIAN KUALITAS H-6")

            print(
                tabel_kualitas_uji.to_string(index=False)
            )


        else:

            print(
                "\nTidak ada judul yang berhasil diambil "
                "dari daftar sitemap eksperimen."
            )


MENCARI SITEMAP INDEX

Memeriksa: https://www.detik.com/sitemap.xml

Sitemap index berhasil: https://www.detik.com/sitemap.xml

SITEMAP NEWS BAGIAN LAIN YANG DITEMUKAN
https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml
https://finance.detik.com/sosok/sitemap_news.xml
https://finance.detik.com/finansial/sitemap_news.xml
https://finance.detik.com/solusiukm/sitemap_news.xml
https://finance.detik.com/perencanaan-keuangan/sitemap_news.xml
https://finance.detik.com/energi/sitemap_news.xml
https://finance.detik.com/industri/sitemap_news.xml
https://finance.detik.com/fintech/sitemap_news.xml
https://finance.detik.com/loker/sitemap_news.xml
https://finance.detik.com/infrastruktur/sitemap_news.xml
https://finance.detik.com/bursa-dan-valas/sitemap_news.xml
https://finance.detik.com/moneter/sitemap_news.xml
https://finance.detik.com/market-research/sitemap_news.xml
https://finance.detik.com/kamus/sitemap_news.xml
https://finance.detik.com/ekonomi-hijau/sitemap_news.xml
https://hot.de

Kode ini menerapkan alur kerja MapReduce menggunakan skrip mapper_kata_kunci.py dan reducer_kata_kunci.py yang dieksekusi via subprocess untuk menghitung frekuensi kata kunci (meta keywords) dari artikel hasil crawling berformat JSONL. Setelah melalui proses ekstraksi kata kunci berpemisah koma, pembersihan spasi, pengurutan (shuffle-sort), dan agregasi reducer, 10 kata kunci terbanyak ditampilkan lalu seluruh hasilnya disimpan ke dalam berkas hasil_frekuensi_kata_kunci.csv.

In [ ]:

# ============================================================
# N.7 WORD COUNT KATA KUNCI
# ============================================================

import subprocess
import sys
import json

# Memilih sumber data hasil crawling
if hasil_crawl_sitemap_lain:

    DATA_KATA_KUNCI = hasil_crawl_sitemap_lain
    SUMBER_KATA_KUNCI = "Sitemap alternatif (Cell 6)"

elif "sampel" in globals() and sampel:

    DATA_KATA_KUNCI = sampel
    SUMBER_KATA_KUNCI = "Hasil crawling H-4"

else:

    DATA_KATA_KUNCI = []
    SUMBER_KATA_KUNCI = "Tidak tersedia"


print("Sumber data:", SUMBER_KATA_KUNCI)
print("Jumlah artikel:", len(DATA_KATA_KUNCI))


# ------------------------------------------------------------
# 1. MEMBUAT FILE MASUKAN JSONL
# ------------------------------------------------------------

Path("masukan_kata_kunci.jsonl").write_text(
    "\n".join(
        json.dumps(x, ensure_ascii=False)
        for x in DATA_KATA_KUNCI
    ) + ("\n" if DATA_KATA_KUNCI else ""),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 2. MEMBUAT MAPPER
# Menghasilkan pasangan (kata_kunci, 1)
# ------------------------------------------------------------

kode_mapper_keyword = r'''
import sys
import json

for baris in sys.stdin:

    try:
        data = json.loads(baris)

    except json.JSONDecodeError:
        continue

    kata_kunci = data.get("kata_kunci", "")

    # Memisahkan kata kunci berdasarkan koma
    for kata in kata_kunci.split(","):

        kata = " ".join(kata.lower().split())

        if kata:
            print(f"{kata}\t1")
'''

Path("mapper_kata_kunci.py").write_text(
    kode_mapper_keyword,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3. MEMBUAT REDUCER
# Menjumlahkan frekuensi setiap kata kunci
# ------------------------------------------------------------

kode_reducer_keyword = r'''
import sys

kunci_sekarang = None
total = 0

for baris in sys.stdin:

    bagian = baris.rstrip("\n").split("\t", 1)

    if len(bagian) != 2:
        continue

    kunci, nilai = bagian

    if kunci != kunci_sekarang:

        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\t{total}")

        kunci_sekarang = kunci
        total = 0

    total += int(nilai)


if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\t{total}")
'''

Path("reducer_kata_kunci.py").write_text(
    kode_reducer_keyword,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. MENJALANKAN MAPPER
# ------------------------------------------------------------

input_keyword = Path(
    "masukan_kata_kunci.jsonl"
).read_text(encoding="utf-8")


hasil_mapper_keyword = subprocess.run(
    [sys.executable, "mapper_kata_kunci.py"],
    input=input_keyword,
    text=True,
    capture_output=True,
    check=True
)


pasangan_keyword = (
    hasil_mapper_keyword.stdout.splitlines()
)

print("\nJumlah pasangan hasil mapper:")
print(f"{len(pasangan_keyword):,}")


# ------------------------------------------------------------
# 5. SHUFFLE DAN SORT
# Reducer membutuhkan key yang berurutan
# ------------------------------------------------------------

pasangan_keyword.sort(
    key=lambda x: x.split("\t", 1)[0]
)


# ------------------------------------------------------------
# 6. MENJALANKAN REDUCER
# ------------------------------------------------------------

input_reducer_keyword = (
    "\n".join(pasangan_keyword)
    + ("\n" if pasangan_keyword else "")
)

hasil_reducer_keyword = subprocess.run(
    [sys.executable, "reducer_kata_kunci.py"],
    input=input_reducer_keyword,
    text=True,
    capture_output=True,
    check=True
)


# ------------------------------------------------------------
# 7. MEMBACA HASIL REDUCER
# ------------------------------------------------------------

data_hasil_keyword = []

for baris in hasil_reducer_keyword.stdout.splitlines():

    kata, jumlah = baris.rsplit("\t", 1)

    data_hasil_keyword.append({
        "kata_kunci": kata,
        "frekuensi": int(jumlah)
    })


df_kata_kunci = pd.DataFrame(
    data_hasil_keyword,
    columns=["kata_kunci", "frekuensi"]
)

df_kata_kunci = df_kata_kunci.sort_values(
    "frekuensi",
    ascending=False
)


# ------------------------------------------------------------
# 8. MENAMPILKAN 10 KATA KUNCI TERATAS
# ------------------------------------------------------------

print("\n10 KATA KUNCI PALING SERING MUNCUL")

if not df_kata_kunci.empty:

    print(
        df_kata_kunci.head(10).to_string(index=False)
    )

else:

    print("Tidak ditemukan kata kunci pada data crawling.")


# ------------------------------------------------------------
# 9. MENYIMPAN HASIL
# ------------------------------------------------------------

df_kata_kunci.to_csv(
    "hasil_frekuensi_kata_kunci.csv",
    index=False
)

print("\nFile yang dibuat:")
print("- masukan_kata_kunci.jsonl")
print("- mapper_kata_kunci.py")
print("- reducer_kata_kunci.py")
print("- hasil_frekuensi_kata_kunci.csv")


Sumber data: Sitemap alternatif (Cell 6)
Jumlah artikel: 15

Jumlah pasangan hasil mapper:
69

10 KATA KUNCI PALING SERING MUNCUL
          kata_kunci  frekuensi
         jabodetabek          3
           kebakaran          3
   menteri pertanian          3
 andi amran sulaiman          2
       jakarta utara          2
       tanjung priok          2
asdp indonesia ferry          1
              bisnis          1
             bnr bri          1
             bapanas          1

File yang dibuat:
- masukan_kata_kunci.jsonl
- mapper_kata_kunci.py
- reducer_kata_kunci.py
- hasil_frekuensi_kata_kunci.csv


### **Tugas Mandiri**

Kode ini membangun inverted index berbasis MapReduce dari judul berita pada partisi Parquet untuk memetakan setiap kata unik ke daftar ID dokumen yang memuatnya. Proses diawali dengan penyiapan identitas dokumen (menggunakan ID asli atau membuat ID sintetis berbasis hash SHA-1 jika kosong), dilanjutkan dengan tahap mapper untuk mengekstrak kata-kata unik per judul, serta tahap shuffle-reduce menggunakan defaultdict(set) untuk mengelompokkan ID dokumen secara terurut. Terakhir, skrip menganalisis statistik sebaran ID (asli vs. sintetis), menampilkan kata-kata dengan cakupan dokumen terbanyak, dan menyimpan inverted index beserta metadatanya ke direktori hasil_inverted_index dalam format JSON, CSV, dan Parquet.

In [ ]:

# ============================================================
# N.8 INVERTED INDEX DENGAN MAPREDUCE
# Mapper  : menghasilkan pasangan (kata, id)
# Reducer : menggabungkan ID berdasarkan kata
# ============================================================

import hashlib
import json
from pathlib import Path
from collections import defaultdict


# ------------------------------------------------------------
# 1. KONFIGURASI
# ------------------------------------------------------------

# False = semua kata diindeks, termasuk stopword.
# True  = kata yang ada di STOPWORD tidak dimasukkan.

INDEX_PAKAI_STOPWORD = False


# ------------------------------------------------------------
# 2. MEMBUAT ID DOKUMEN
# ID asli digunakan jika tersedia.
# Jika kosong, dibuat ID sintetis.
# ------------------------------------------------------------

def buat_id_dokumen(id_asli, kategori, judul, tanggal):

    # Menggunakan ID asli jika tersedia
    nilai_id = pd.to_numeric(
        id_asli,
        errors="coerce"
    )

    if pd.notna(nilai_id):

        return str(int(nilai_id)), "ID asli"

    # Membuat ID sintetis jika ID asli kosong
    tanggal_teks = (
        str(tanggal)
        if pd.notna(tanggal)
        else ""
    )

    identitas = (
        f"{kategori}|{tanggal_teks}|{judul}"
    )

    kode_hash = hashlib.sha1(
        identitas.encode("utf-8")
    ).hexdigest()[:12]

    id_sintetis = (
        f"{kategori}_tanpa_id_{kode_hash}"
    )

    return id_sintetis, "ID sintetis"


# ------------------------------------------------------------
# 3. MAPPER
# Membaca setiap file Parquet sebagai input split.
# Menghasilkan pasangan (kata, id).
# ------------------------------------------------------------

def mapper_inverted_index(path):

    df_tmp = pd.read_parquet(
        path,
        columns=["id", "judul", "tanggal"]
    )

    # Mengambil kategori dari nama folder partisi
    kategori = path.parent.name.split("=", 1)[1]

    pasangan = []
    metadata = []

    for _, row in df_tmp.iterrows():

        judul = row["judul"]

        if pd.isna(judul):
            continue

        # Membuat atau mengambil ID artikel
        doc_id, jenis_id = buat_id_dokumen(
            row["id"],
            kategori,
            judul,
            row["tanggal"]
        )

        # Tokenisasi judul
        daftar_kata = tokenisasi(judul)

        # Menghapus stopword jika opsi diaktifkan
        if INDEX_PAKAI_STOPWORD:

            daftar_kata = [
                kata for kata in daftar_kata
                if kata not in STOPWORD
            ]

        # Satu kata hanya menghasilkan satu pasangan
        # untuk setiap judul, meskipun muncul berulang.
        kata_unik = sorted(set(daftar_kata))

        for kata in kata_unik:

            pasangan.append(
                (kata, doc_id)
            )

        # Menyimpan informasi dokumen
        metadata.append({
            "doc_id": doc_id,
            "id_asli": (
                str(int(float(row["id"])))
                if pd.notna(row["id"])
                else None
            ),
            "jenis_id": jenis_id,
            "kategori": kategori,
            "judul": judul,
            "tanggal": str(row["tanggal"])
        })

    return pasangan, metadata


# ------------------------------------------------------------
# 4. MAP PHASE
# Menjalankan mapper pada seluruh input split.
# ------------------------------------------------------------

hasil_mapper_index = []

metadata_dokumen = []

for path in splits:

    pasangan, metadata = mapper_inverted_index(path)

    hasil_mapper_index.append(pasangan)

    metadata_dokumen.extend(metadata)


jumlah_pasangan_mapper = sum(
    len(x) for x in hasil_mapper_index
)


print("MAP PHASE SELESAI")

print("Jumlah mapper:", len(hasil_mapper_index))

print(
    "Total pasangan (kata, id):",
    f"{jumlah_pasangan_mapper:,}"
)


# ------------------------------------------------------------
# 5. SHUFFLE DAN REDUCE
# Mengelompokkan seluruh ID berdasarkan kata.
# Menggunakan set agar ID tidak berulang.
# ------------------------------------------------------------

shuffle_index = defaultdict(set)

for hasil_mapper in hasil_mapper_index:

    for kata, doc_id in hasil_mapper:

        shuffle_index[kata].add(doc_id)


# Reducer menghasilkan:
# kata -> daftar ID judul yang memuat kata tersebut

inverted_index = {

    kata: sorted(daftar_id)

    for kata, daftar_id in sorted(
        shuffle_index.items()
    )
}


print("\nREDUCE PHASE SELESAI")

print(
    "Jumlah kata unik:",
    f"{len(inverted_index):,}"
)


# ------------------------------------------------------------
# 6. MEMERIKSA ID SINTETIS
# Khususnya artikel kategori inet.
# ------------------------------------------------------------

df_metadata_index = pd.DataFrame(
    metadata_dokumen
)

jumlah_id_asli = (
    df_metadata_index["jenis_id"] == "ID asli"
).sum()

jumlah_id_sintetis = (
    df_metadata_index["jenis_id"] == "ID sintetis"
).sum()

print("\nSTATISTIK ID DOKUMEN")

print("Jumlah dokumen:",
      f"{len(df_metadata_index):,}")

print("ID asli:",
      f"{jumlah_id_asli:,}")

print("ID sintetis:",
      f"{jumlah_id_sintetis:,}")


print("\nSEBARAN ID BERDASARKAN KATEGORI")

print(
    pd.crosstab(
        df_metadata_index["kategori"],
        df_metadata_index["jenis_id"]
    ).to_string()
)


# Memeriksa apakah doc_id mengalami duplikasi
jumlah_duplikat_id = (
    df_metadata_index["doc_id"].duplicated().sum()
)

print("\nID dokumen duplikat:",
      jumlah_duplikat_id)


# ------------------------------------------------------------
# 7. MENAMPILKAN CONTOH INVERTED INDEX
# ------------------------------------------------------------

print("\nCONTOH INVERTED INDEX")

kata_contoh = [
    "corona",
    "virus",
    "berita",
    "indonesia"
]

for kata in kata_contoh:

    daftar_id = inverted_index.get(kata, [])

    print(f"\nKata: {kata}")
    print("Jumlah judul:", len(daftar_id))
    print("Daftar ID:", daftar_id[:20])

    if len(daftar_id) > 20:
        print("... dan", len(daftar_id) - 20, "ID lainnya")


# ------------------------------------------------------------
# 8. MENAMPILKAN 10 KATA DENGAN DOKUMEN TERBANYAK
# ------------------------------------------------------------

peringkat_index = sorted(
    inverted_index.items(),
    key=lambda x: len(x[1]),
    reverse=True
)

print("\n10 KATA DENGAN JUMLAH DOKUMEN TERBANYAK")

for kata, daftar_id in peringkat_index[:10]:

    print(
        f"{kata:<15} {len(daftar_id):>8,} dokumen"
    )


# ------------------------------------------------------------
# 9. MENYIMPAN HASIL INVERTED INDEX
# ------------------------------------------------------------

FOLDER_INDEX = Path("hasil_inverted_index")
FOLDER_INDEX.mkdir(exist_ok=True)


# Menyimpan inverted index dalam JSON
with open(
    FOLDER_INDEX / "inverted_index.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        inverted_index,
        file,
        ensure_ascii=False,
        indent=2
    )


# Membuat tabel kata dan daftar ID
df_hasil_index = pd.DataFrame([

    {
        "kata": kata,
        "jumlah_dokumen": len(daftar_id),
        "daftar_id": json.dumps(
            daftar_id,
            ensure_ascii=False
        )
    }

    for kata, daftar_id in inverted_index.items()

])


# Menyimpan hasil sebagai CSV dan Parquet
df_hasil_index.to_csv(
    FOLDER_INDEX / "inverted_index.csv",
    index=False
)

df_hasil_index.to_parquet(
    FOLDER_INDEX / "inverted_index.parquet",
    index=False
)


# Menyimpan metadata ID untuk menelusuri judul
df_metadata_index.to_csv(
    FOLDER_INDEX / "metadata_dokumen.csv",
    index=False
)


# ------------------------------------------------------------
# 10. RINGKASAN AKHIR
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("INVERTED INDEX SELESAI")
print("=" * 65)

print("Jumlah dokumen:",
      f"{len(df_metadata_index):,}")

print("Jumlah kata unik:",
      f"{len(inverted_index):,}")

print("Jumlah pasangan mapper:",
      f"{jumlah_pasangan_mapper:,}")

print("Jumlah ID sintetis:",
      f"{jumlah_id_sintetis:,}")

print("\nFile yang berhasil disimpan:")

print("- hasil_inverted_index/inverted_index.json")
print("- hasil_inverted_index/inverted_index.csv")
print("- hasil_inverted_index/inverted_index.parquet")
print("- hasil_inverted_index/metadata_dokumen.csv")

print("=" * 65)


MAP PHASE SELESAI
Jumlah mapper: 9
Total pasangan (kata, id): 817,993

REDUCE PHASE SELESAI
Jumlah kata unik: 32,275

STATISTIK ID DOKUMEN
Jumlah dokumen: 90,818
ID asli: 85,241
ID sintetis: 5,577

SEBARAN ID BERDASARKAN KATEGORI
jenis_id  ID asli  ID sintetis
kategori                      
finance     14141            0
food         4314            0
health       4914            0
hot         16290            0
inet            0         5577
news        32321            0
oto          4375            0
sport        2433            0
travel       6453            0

ID dokumen duplikat: 0

CONTOH INVERTED INDEX

Kata: corona
Jumlah judul: 17228
Daftar ID: ['4856495', '4856598', '4868317', '4868545', '4869163', '4869372', '4869504', '4870158', '4870590', '4870603', '4870636', '4870642', '4870903', '4870973', '4871002', '4871504', '4871515', '4871658', '4871758', '4871792']
... dan 17208 ID lainnya

Kata: virus
Jumlah judul: 3067
Daftar ID: ['4856495', '4856598', '4860201', '4865102', '48

In [ ]:

# ============================================================
# N.9 BACKUP LAPISAN DATA KE GOOGLE DRIVE
#
# Isi ZIP:
# 1. lapisan_mentah    -> contoh maksimal 100 baris JSONL
# 2. lapisan_terkurasi -> seluruh file Parquet
# 3. siap_pakai        -> seluruh hasil analisis
#
# ============================================================

import os
import json
import zipfile
from pathlib import Path
from datetime import datetime


# ------------------------------------------------------------
# 1. MENGHUBUNGKAN GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. MENENTUKAN LOKASI FOLDER
# ------------------------------------------------------------

FOLDER_MENTAH = Path("lapisan_mentah")
FOLDER_TERKURASI = Path("lapisan_terkurasi")
FOLDER_SIAP = Path("siap_pakai")

# Lokasi penyimpanan di Google Drive
FOLDER_GOOGLE_DRIVE = Path(
    "/content/drive/MyDrive/Praktikum_Big_Data_5"
)

FOLDER_GOOGLE_DRIVE.mkdir(
    parents=True,
    exist_ok=True
)


# Nama ZIP menggunakan tanggal dan waktu
waktu_backup = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

NAMA_ZIP = (
    f"backup_lapisan_data_{waktu_backup}.zip"
)

LOKASI_ZIP = (
    FOLDER_GOOGLE_DRIVE / NAMA_ZIP
)


# ------------------------------------------------------------
# 3. MEMBUAT ZIP
# ------------------------------------------------------------

jumlah_file = {
    "lapisan_mentah": 0,
    "lapisan_terkurasi": 0,
    "siap_pakai": 0
}

ukuran_asli = 0

with zipfile.ZipFile(
    LOKASI_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zipf:

    # --------------------------------------------------------
    # A. LAPISAN MENTAH
    # Hanya mengambil maksimal 100 baris JSONL
    # --------------------------------------------------------

    print("Mengemas lapisan_mentah...")

    zipf.writestr(
        "lapisan_mentah/",
        ""
    )

    if FOLDER_MENTAH.exists():

        daftar_jsonl = sorted(
            FOLDER_MENTAH.glob("*.jsonl")
        )

        jumlah_sampel = 0
        BATAS_SAMPEL = 100

        baris_sampel = []

        for file in daftar_jsonl:

            if jumlah_sampel >= BATAS_SAMPEL:
                break

            with open(
                file,
                "r",
                encoding="utf-8"
            ) as f:

                for baris in f:

                    if jumlah_sampel >= BATAS_SAMPEL:
                        break

                    if baris.strip():

                        baris_sampel.append(
                            baris.rstrip("\n")
                        )

                        jumlah_sampel += 1

        # Menyimpan contoh data mentah
        if baris_sampel:

            isi_sampel = "\n".join(baris_sampel) + "\n"

            zipf.writestr(
                "lapisan_mentah/contoh_100_baris.jsonl",
                isi_sampel
            )

            jumlah_file["lapisan_mentah"] = 1

        zipf.writestr(
            "lapisan_mentah/README.txt",
            (
                "Folder ini hanya berisi contoh data mentah "
                "sebanyak maksimal 100 baris JSONL.\n"
                "Seluruh data mentah asli tidak dimasukkan "
                "ke dalam ZIP untuk menghemat ukuran penyimpanan.\n"
            )
        )

        print(
            "  Contoh data mentah:",
            jumlah_sampel,
            "baris"
        )

    else:

        print("  Folder lapisan_mentah tidak ditemukan.")


    # --------------------------------------------------------
    # B. LAPISAN TERKURASI
    # Memasukkan seluruh file hasil cleaning
    # --------------------------------------------------------

    print("\nMengemas lapisan_terkurasi...")

    if FOLDER_TERKURASI.exists():

        for file in FOLDER_TERKURASI.rglob("*"):

            if file.is_file():

                nama_dalam_zip = (
                    "lapisan_terkurasi/"
                    + file.relative_to(
                        FOLDER_TERKURASI
                    ).as_posix()
                )

                zipf.write(
                    file,
                    arcname=nama_dalam_zip
                )

                jumlah_file["lapisan_terkurasi"] += 1

                ukuran_asli += file.stat().st_size

        print(
            "  Jumlah file:",
            jumlah_file["lapisan_terkurasi"]
        )

    else:

        zipf.writestr(
            "lapisan_terkurasi/",
            ""
        )

        print("  Folder lapisan_terkurasi tidak ditemukan.")


    # --------------------------------------------------------
    # C. SIAP PAKAI
    # Memasukkan Parquet dan database SQLite
    # --------------------------------------------------------

    print("\nMengemas siap_pakai...")

    if FOLDER_SIAP.exists():

        for file in FOLDER_SIAP.rglob("*"):

            if file.is_file():

                nama_dalam_zip = (
                    "siap_pakai/"
                    + file.relative_to(
                        FOLDER_SIAP
                    ).as_posix()
                )

                zipf.write(
                    file,
                    arcname=nama_dalam_zip
                )

                jumlah_file["siap_pakai"] += 1

                ukuran_asli += file.stat().st_size

        print(
            "  Jumlah file:",
            jumlah_file["siap_pakai"]
        )

    else:

        zipf.writestr(
            "siap_pakai/",
            ""
        )

        print("  Folder siap_pakai tidak ditemukan.")


    # --------------------------------------------------------
    # D. README ARSIP
    # --------------------------------------------------------

    isi_readme = f"""
BACKUP LAPISAN DATA - PRAKTIKUM BIG DATA 5

Tanggal backup:
{datetime.now().strftime("%d-%m-%Y %H:%M:%S")}

STRUKTUR ARSIP:

1. lapisan_mentah/
   Berisi contoh maksimal 100 baris data mentah JSONL.
   Data mentah lengkap tidak disertakan untuk menghemat ukuran.

2. lapisan_terkurasi/
   Berisi seluruh hasil data cleaning dalam format Parquet.
   Data disimpan berdasarkan partisi kategori.

3. siap_pakai/
   Berisi hasil analisis yang siap digunakan.
   Termasuk hasil Parquet dan database SQLite.

JUMLAH FILE:

Lapisan mentah     : {jumlah_file["lapisan_mentah"]}
Lapisan terkurasi  : {jumlah_file["lapisan_terkurasi"]}
Siap pakai         : {jumlah_file["siap_pakai"]}

Catatan:
Arsip ini dibuat sebagai backup hasil praktikum.
Data asli pada notebook tidak dipindahkan atau dihapus.
"""

    zipf.writestr(
        "README_arsip.txt",
        isi_readme
    )


# ------------------------------------------------------------
# 4. INFORMASI FILE ZIP
# ------------------------------------------------------------

ukuran_zip = LOKASI_ZIP.stat().st_size

print("\n" + "=" * 60)
print("BACKUP GOOGLE DRIVE BERHASIL DIBUAT")
print("=" * 60)

print("\nLokasi penyimpanan:")
print(LOKASI_ZIP)

print("\nNama file:")
print(NAMA_ZIP)

print("\nUkuran ZIP:")
print(round(ukuran_zip / (1024 * 1024), 2), "MB")

print("\nIsi arsip:")

for nama, jumlah in jumlah_file.items():

    print(
        f"{nama:<22}: {jumlah} file"
    )

print("\nData mentah:")
print("Hanya contoh maksimal 100 baris.")

print("\nData asli:")
print("Tidak ada folder asli yang dihapus atau dipindahkan.")

print("\n" + "=" * 60)
print("SELESAI")
print("=" * 60)


Mounted at /content/drive
Mengemas lapisan_mentah...
  Contoh data mentah: 100 baris

Mengemas lapisan_terkurasi...
  Jumlah file: 9

Mengemas siap_pakai...
  Jumlah file: 2

BACKUP GOOGLE DRIVE BERHASIL DIBUAT

Lokasi penyimpanan:
/content/drive/MyDrive/Praktikum_Big_Data_5/backup_lapisan_data_20261108_225949.zip

Nama file:
backup_lapisan_data_20261108_225949.zip

Ukuran ZIP:
8.02 MB

Isi arsip:
lapisan_mentah        : 1 file
lapisan_terkurasi     : 9 file
siap_pakai            : 2 file

Data mentah:
Hanya contoh maksimal 100 baris.

Data asli:
Tidak ada folder asli yang dihapus atau dipindahkan.

SELESAI
